# Trích xuất feature ONE-PEACE (video + audio) cho YouCookII — Colab / Kaggle / máy local

Chỉ sửa **ô 1**. Ô 2 tự nhận môi trường và chuẩn bị mọi thứ.

- **Chạy tiếp khi bị ngắt:** chạy lại ô 1, 2 rồi ô chạy thật — video có trong `done_*.txt` hoặc đã có `.npy` sẽ được bỏ qua.
- **Colab:** `.npy` ghi vào `/content/feats_video`, `/content/feats_audio`; chỉ `done_*.txt` (vài chục KB) nằm trên Drive.
  ⚠ Phiên tắt là mất `.npy` trong `/content` — lấy ra trước (ô **Z**). Nếu `.npy` đã mất mà txt vẫn ghi "đã xong",
  xoá các dòng đó trong txt để làm lại.
- **Kaggle:** *Accelerator = GPU T4 x2*, *Internet = On*, upload video + checkpoint làm Dataset. Tự dùng cả 2 GPU.
  Chạy bằng **Save Version → Save & Run All** để giữ `/kaggle/working`.
- **Local:** dùng `models/`, `venv/`, `op310/`, `ONE-PEACE/` có sẵn; điền `video_dir` (video đặt tên `<id YouCookII>.mp4`).
- **`DATASET`:** `"youcookii"` hoặc `"desed"` — tự chọn annotation, stride (8 / 0.5 s hoặc 4 / 0.25 s) và thư mục output riêng.
  Nhớ trỏ `video_dir` tới đúng video của bộ đó (DESED: clip 10 s, tên `<id>_<start>_<end>.mp4`).
- **`TEST_MODE = True`:** 3 video, tối đa 16 clip mỗi video, ghi ra `*_test` (không đụng done list thật).

In [12]:
# 1. CẤU HÌNH — chỉ sửa ô này
DATASET = "desed"        # "youcookii" (stride 8 = 0.5 s) | "desed" (stride 4 = 0.25 s, clip 10 s)
TEST_MODE = True            # True: chạy thử 3 video / 16 clip vào thư mục *_test
NUM_SHARDS, SHARD_ID = 1, 0  # chạy song song nhiều nơi (vd. Colab + Kaggle): NUM_SHARDS = 2, mỗi nơi một SHARD_ID
BATCH_VIDEO = 16             # A100: 32 | T4: 16 | RTX 3060: 8
DONE_EXTRA = []              # (tuỳ chọn) thêm file done .txt / thư mục .npy đã làm ở nơi khác

DRIVE = "/content/drive/MyDrive/code KL"
PATHS = {
    "colab": dict(
        video_dir  = f"{DRIVE}/Omni/data/YouCookII/videos",
        video_ckpt = f"{DRIVE}/One_Peace/onepeace_video_k400.pth",   # tự copy ra /content trước khi chạy
        audio_ckpt = f"{DRIVE}/One_Peace/one-peace-audio.pt",
        out_dir    = "/content",                  # -> /content/feats_video, /content/feats_audio
        done_dir   = f"{DRIVE}/feats_done",       # chỉ done_*.txt nằm trên Drive
    ),
    "kaggle": dict(
        video_dir  = "/kaggle/input/youcookii-videos/videos",                # SỬA theo tên Dataset
        video_ckpt = "/kaggle/input/onepeace-ckpt/onepeace_video_k400.pth",  # SỬA
        audio_ckpt = "/kaggle/input/onepeace-ckpt/one-peace-audio.pt",       # SỬA
        out_dir    = "/kaggle/working",
        done_dir   = None,                        # None = done txt nằm trong thư mục output
    ),
    "local": dict(
        video_dir  = r"samples\test_video",                         # ĐIỀN: thư mục có vài video <id>.mp4
        video_ckpt = r"models\onepeace_video_k400.pth",
        audio_ckpt = r"models\one-peace-audio.pt",
        out_dir    = r"output",
        done_dir   = None,
    ),
}
LOCAL_ROOT = r"."   # chứa One_Peace_FE, ONE-PEACE, venv, op310

In [13]:
# 2. THIẾT LẬP — không cần sửa: nhận môi trường, suy ra đường dẫn, lấy code + thư viện
import os, shutil

# thư mục chứa notebook: đường dẫn tương đối ở ô 1 tính từ đây (không phụ thuộc %cd của các ô sau)
def _find_nb_dir():
    nb_file = globals().get("__vsc_ipynb_file__")        # VS Code cho biết đường dẫn notebook
    if nb_file:
        return os.path.dirname(nb_file)
    d = os.getcwd()
    while True:                                          # Jupyter: đi lên tới thư mục có EXTRACT.ipynb
        if os.path.exists(os.path.join(d, "EXTRACT.ipynb")):
            return d
        if os.path.dirname(d) == d:
            return os.getcwd()
        d = os.path.dirname(d)
NB_DIR = _find_nb_dir()
def absp(p):
    return p if not p or os.path.isabs(p) else os.path.normpath(os.path.join(NB_DIR, p))

ENV = ("kaggle" if os.path.exists("/kaggle/input")
       else "colab" if "COLAB_RELEASE_TAG" in os.environ or os.path.isdir("/content")
       else "local")
P = {k: absp(v) for k, v in PATHS[ENV].items()}
WORK = {"colab": "/content", "kaggle": "/tmp/work", "local": absp(LOCAL_ROOT)}[ENV]   # repo + venv
REPO, ONEPEACE_REPO = os.path.join(WORK, "One_Peace_FE"), os.path.join(WORK, "ONE-PEACE")
if ENV == "local":
    PY_VIDEO = os.path.join(WORK, "venv", "Scripts", "python.exe")
    PY_AUDIO = os.path.join(WORK, "op310", "Scripts", "python.exe")
else:
    PY_VIDEO, PY_AUDIO = "python", f"{WORK}/op310/bin/python"

if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
assert P["video_dir"] and os.path.isdir(P["video_dir"]), f"Không thấy thư mục video: {P['video_dir']!r} (sửa ở ô 1)"
VIDEO_DIR = P["video_dir"]

# annotation + bước trượt theo bộ dữ liệu (theo README của UniAV)
DS = {
    "youcookii": dict(anno="annotations/youcookii_all.json", stride=8, stride_sec=0.5),
    "desed":     dict(anno="annotations/dcase_all.json",     stride=4, stride_sec=0.25),
}[DATASET]
ANNO, STRIDE, STRIDE_SEC = DS["anno"], DS["stride"], DS["stride_sec"]

# youcookii giữ tên thư mục cũ (feats_video); dataset khác thêm hậu tố để không lẫn
suffix = ("" if DATASET == "youcookii" else f"_{DATASET}") + ("_test" if TEST_MODE else "")
OUT_VIDEO = os.path.join(P["out_dir"], "feats_video" + suffix)
OUT_AUDIO = os.path.join(P["out_dir"], "feats_audio" + suffix)
DONE_DIR = None if TEST_MODE else P["done_dir"]
for d in [WORK, OUT_VIDEO, OUT_AUDIO] + ([DONE_DIR] if DONE_DIR else []):
    os.makedirs(d, exist_ok=True)

# tham số chung thêm vào lệnh chạy: test / compile / done list
done = ([f'--done_dir "{DONE_DIR}"'] if DONE_DIR else []) + (
    ["--done_list"] + [f'"{p}"' for p in DONE_EXTRA] if DONE_EXTRA and not TEST_MODE else [])
# TEST_MODE: 3 video; YouCookII thêm giới hạn 16 clip/video (clip DESED chỉ 10 s nên để đủ 37 bước)
test_video = "--limit 3" + (" --max_clips 16" if DATASET == "youcookii" else "")
EXTRA_VIDEO = " ".join(([test_video] if TEST_MODE else ["--compile"]) + done)
EXTRA_AUDIO = " ".join((["--limit 3"] if TEST_MODE else []) + done)

def local_ckpt(path):
    """Colab: copy checkpoint từ Drive ra /content (nạp nhanh hơn); nơi khác dùng thẳng."""
    if ENV == "colab" and path.startswith("/content/drive"):
        dst = os.path.join("/content", os.path.basename(path))
        if not os.path.exists(dst):
            print("copy", path, "->", dst)
            shutil.copy(path, dst)
        path = dst
    print(path, os.path.getsize(path), "byte")
    return path

def n_npy(d):
    return sum(f.endswith(".npy") for f in os.listdir(d)) if os.path.isdir(d) else 0

if ENV != "local":
    !git clone -q https://github.com/chien24/One_Peace_FE.git "{REPO}" 2>/dev/null || git -C "{REPO}" pull -q
    !pip install -q einops imageio-ffmpeg
print(f"Môi trường: {ENV} | dataset: {DATASET} (stride {STRIDE} / {STRIDE_SEC} s){' | TEST_MODE' if TEST_MODE else ''} | CPU: {os.cpu_count()}")
print("output:", OUT_VIDEO, "|", OUT_AUDIO, "| done txt:", DONE_DIR or "(trong output)")
!nvidia-smi -L

Môi trường: local | dataset: desed (stride 4 / 0.25 s) | TEST_MODE | CPU: 12
output: c:\Users\PC\Downloads\UNI_extract\output\feats_video_desed_test | c:\Users\PC\Downloads\UNI_extract\output\feats_audio_desed_test | done txt: (trong output)


GPU 0: NVIDIA GeForce RTX 3060 (UUID: GPU-78754f43-1b1e-826e-775e-783aeb76de2e)


## VIDEO
Log in `TB ... clip/s | chờ giải mã X%`: nếu X lớn (> 10%) thì CPU giải mã là nút thắt — thêm `--decode_workers N`;
nếu X ≈ 0 thì GPU đã chạy hết công suất.

In [14]:
# 3. Trích xuất video. Bị ngắt thì chạy lại ô 1, 2 rồi ô này: video đã xong sẽ được bỏ qua.
VIDEO_CKPT = local_ckpt(P["video_ckpt"])
%cd {REPO}
!"{PY_VIDEO}" extract_video_features.py --checkpoint "{VIDEO_CKPT}" --video_dir "{VIDEO_DIR}" --output_dir "{OUT_VIDEO}" \
    --ids_from {ANNO} --stride {STRIDE} --batch_size {BATCH_VIDEO} \
    --num_shards {NUM_SHARDS} --shard_id {SHARD_ID} {EXTRA_VIDEO}

c:\Users\PC\Downloads\UNI_extract\models\onepeace_video_k400.pth 6629055726 byte
c:\Users\PC\Downloads\UNI_extract\One_Peace_FE
1 video trong shard 0/1, đã xong 1, còn 0 video cần xử lý | GPU: [0]


## AUDIO — chạy sau khi xong video, hoặc ở một phiên riêng
Không chạy video và audio cùng lúc trên cùng GPU: hai model không vừa VRAM.

In [ ]:
# A1. Môi trường Python 3.10 riêng cho fairseq cũ của ONE-PEACE (~5 phút). Local: đã có op310, bỏ qua.
if ENV != "local":
    !git clone -q --depth 1 https://github.com/OFA-Sys/ONE-PEACE "{ONEPEACE_REPO}" 2>/dev/null || echo "ONE-PEACE đã có"
    !pip install -q uv && uv venv -q --seed --python 3.10 "{WORK}/op310"
    !"{PY_AUDIO}" -m pip install -q "pip==24.0"
    # torch và torchvision phải cài CÙNG một lệnh, nếu không timm sẽ kéo về bản torch CPU
    !"{PY_AUDIO}" -m pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
    # setuptools<81: bản mới bỏ pkg_resources mà librosa 0.10 vẫn import. resampy: cho --res_type kaiser_best
    !"{PY_AUDIO}" -m pip install -q "setuptools<81" "numpy<2" hydra-core==1.0.7 omegaconf==2.0.6 \
        antlr4-python3-runtime==4.8 bitarray sacrebleu tabulate regex timm==0.6.11 iopath tensorboardX pydub \
        librosa==0.10.0 resampy soundfile soxr einops opencv-python-headless scipy tqdm pillow imageio-ffmpeg
# phải in: torch 2.1.2+cu121 cuda True
!"{PY_AUDIO}" -c "import torch, librosa, resampy; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'GPU')"

In [ ]:
# A2. Trích xuất audio. Bị ngắt thì chạy lại ô 1, 2, A1 rồi ô này: video đã xong sẽ được bỏ qua.
# STRIDE_SEC = STRIDE / 16 (khớp bước của visual). Mặc định: dtype fp16, res_type kaiser_best.
AUDIO_CKPT = local_ckpt(P["audio_ckpt"])   # phải là 6139030001 byte
%cd {REPO}
!"{PY_AUDIO}" extract_audio_features.py --onepeace_repo "{ONEPEACE_REPO}" \
    --checkpoint "{AUDIO_CKPT}" --video_dir "{VIDEO_DIR}" --output_dir "{OUT_AUDIO}" \
    --ids_from {ANNO} --stride_sec {STRIDE_SEC} --batch_size 64 \
    --num_shards {NUM_SHARDS} --shard_id {SHARD_ID} {EXTRA_AUDIO}

## KIỂM TRA + ZIP

In [ ]:
# C. Đếm file và so số bước visual / audio
print("video:", n_npy(OUT_VIDEO), "| audio:", n_npy(OUT_AUDIO))
%cd {REPO}
!"{PY_VIDEO}" check_features.py --anno {ANNO} --feat_dir "{OUT_VIDEO}" --audio_feat_dir "{OUT_AUDIO}" --stride {STRIDE}
# Mong đợi: |T_visual - T_audio| = 0-2 bước, chuẩn L2 visual ~20-33, audio ~1.000.
# "thiếu file" > 0 nếu còn shard chưa chạy hoặc có video lỗi (xem failed_*.txt trong thư mục output).

In [ ]:
# Z. (Tuỳ chọn) Zip toàn bộ thư mục output (ghi đè zip cũ) -> feats_video.zip, feats_audio.zip cạnh thư mục
for d in [OUT_VIDEO, OUT_AUDIO]:
    if n_npy(d):
        zip_path = shutil.make_archive(d, "zip", root_dir=d)
        print(zip_path, f"{os.path.getsize(zip_path) / 2**30:.2f} GB, {n_npy(d)} file .npy")